In [1]:
import pandas as pd
import numpy as np
import joblib

from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import SaltRemover
from rdkit.Chem.MolStandardize import rdMolStandardize

from mordred import Calculator, descriptors

In [2]:
pipeline = joblib.load("lgbm_pipeline.pkl")

final_feature_columns = joblib.load(
    "final_feature_columns.pkl"
)


In [3]:
remover = SaltRemover.SaltRemover()

largest_fragment_chooser = (
    rdMolStandardize.LargestFragmentChooser()
)

def clean_molecule(smiles):

    try:

        mol = Chem.MolFromSmiles(smiles)

        if mol is None:
            return None

        mol = remover.StripMol(mol)

        mol = largest_fragment_chooser.choose(
            mol
        )

        Chem.SanitizeMol(mol)

        return mol

    except:
        return None

In [4]:
calc = Calculator(
    descriptors,
    ignore_3D=True
)

def generate_descriptors(mol):

    desc_df = calc.pandas([mol])

    desc_df = desc_df.apply(
        pd.to_numeric,
        errors="coerce"
    )

    desc_df.columns = [
        f"DESC_{c}"
        for c in desc_df.columns
    ]

    return desc_df

In [5]:
def generate_fingerprint(mol):

    fp = list(
        AllChem.GetMorganFingerprintAsBitVect(
            mol,
            radius=2,
            nBits=1024
        )
    )

    fp_df = pd.DataFrame([fp])

    fp_df.columns = [
        f"FPMorgan_{i}"
        for i in range(1024)
    ]

    return fp_df

In [6]:
def predict_solubility(smiles):

    mol = clean_molecule(smiles)

    if mol is None:
        return np.nan

    descriptor_df = generate_descriptors(mol)

    fingerprint_df = generate_fingerprint(mol)

    X_new = pd.concat(
        [descriptor_df, fingerprint_df],
        axis=1
    )

    X_new = X_new.reindex(
        columns=final_feature_columns,
        fill_value=np.nan
    )

    prediction = pipeline.predict(X_new)

    return prediction[0]